In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('WELFake_Dataset.csv')
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [4]:
df.isnull().sum()

Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

In [5]:
df = df.dropna()
df.isnull().sum()

Unnamed: 0    0
title         0
text          0
label         0
dtype: int64

In [6]:
X = df.drop('label', axis=1)
y = df['label']

In [7]:
X.shape

(71537, 3)

In [9]:
import tensorflow as tf
tf.__version__

'2.16.2'

In [10]:
gpus = tf.config.list_physical_devices('GPU')
print("GPUs:", gpus)
print("Using:", "GPU" if gpus else "CPU")

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Using: GPU


In [36]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Input


In [12]:
### Vocabulary size
voc_size = 5000

### OneHot Representation

In [13]:
messages = X.copy()

In [17]:
messages['title']

0        LAW ENFORCEMENT ON HIGH ALERT Following Threat...
2        UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3        Bobby Jindal, raised Hindu, uses story of Chri...
4        SATAN 2: Russia unvelis an image of its terrif...
5        About Time! Christian Group Sues Amazon and SP...
                               ...                        
72129    Russians steal research on Trump in hack of U....
72130     WATCH: Giuliani Demands That Democrats Apolog...
72131    Migrants Refuse To Leave Train At Refugee Camp...
72132    Trump tussle gives unpopular Mexican leader mu...
72133    Goldman Sachs Endorses Hillary Clinton For Pre...
Name: title, Length: 71537, dtype: str

In [18]:
messages.reset_index(inplace=True)

In [24]:
messages

,index,Unnamed: 0,title,text
0,0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...
1,2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ..."
2,3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...
3,4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will..."
4,5,5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...
...,...,...,...,...
71532,72129,72129,Russians steal research on Trump in hack of U....,WASHINGTON (Reuters) - Hackers believed to be ...
71533,72130,72130,WATCH: Giuliani Demands That Democrats Apolog...,"You know, because in fantasyland Republicans n..."
71534,72131,72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...
71535,72132,72132,Trump tussle gives unpopular Mexican leader mu...,MEXICO CITY (Reuters) - Donald Trump’s combati...


In [26]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))


In [27]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/hossain/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [28]:
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()
    
    review = [ps.stem(word) for word in review if not word in stop_words]
    review = ' '.join(review)
    corpus.append(review)

In [30]:
corpus[0]

'law enforc high alert follow threat cop white blacklivesmatt fyf terrorist video'

In [31]:
messages['title'][0]

'LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]'

In [32]:
onehot_repr = [one_hot(words, voc_size) for words in corpus]
onehot_repr[0]

[4408, 4222, 4777, 4783, 604, 2384, 3094, 324, 4945, 4459, 4850, 3134]

### Embedding Representation

In [33]:
sent_length = 20
embedded_docs = pad_sequences(onehot_repr, padding='pre', maxlen=sent_length)
embedded_docs[0]

array([   0,    0,    0,    0,    0,    0,    0,    0, 4408, 4222, 4777,
       4783,  604, 2384, 3094,  324, 4945, 4459, 4850, 3134], dtype=int32)

### Creating Model

In [37]:
embedding_vector_features = 40 # feature representation for each word
model = Sequential()
model.add(Input(shape=(sent_length,)))
model.add(Embedding(voc_size, embedding_vector_features))
model.add(LSTM(100))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 20, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │        56,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,501 (1001.96 KB)

 Trainable params: 256,501 (1001.96 KB)

 Non-trainable params: 0 (0.00 B)

None


In [38]:
len(embedded_docs), y.shape

(71537, (71537,))

In [39]:
import numpy as np
X_final = np.array(embedded_docs)
y_final = np.array(y)

### Train test split

In [40]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

### Model Training

In [53]:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=64)

Epoch 1/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.8565 - loss: 0.3188 - val_accuracy: 0.8808 - val_loss: 0.2729
Epoch 2/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9060 - loss: 0.2270 - val_accuracy: 0.8977 - val_loss: 0.2460
Epoch 3/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9185 - loss: 0.2017 - val_accuracy: 0.8966 - val_loss: 0.2493
Epoch 4/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.9273 - loss: 0.1835 - val_accuracy: 0.8968 - val_loss: 0.2526
Epoch 5/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9334 - loss: 0.1683 - val_accuracy: 0.8971 - val_loss: 0.2671
Epoch 6/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.9384 - loss: 0.1554 - val_accuracy: 0.8960 - val_loss: 0.2863
Epoch 7/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9447 - loss: 0.1415 - val_accuracy: 0.8944 - val_loss: 0.2808
Epoch 8/10
749/749 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9488 - loss: 0.1305 - val_acc

### Adding Dropout

In [52]:
from tensorflow.keras.layers import Dropout
embedding_vector_features = 40 # feature representation for each word
model = Sequential()
model.add(Input(shape=(sent_length,)))
model.add(Embedding(voc_size, embedding_vector_features))
model.add(Dropout(0.3))
model.add(LSTM(100))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 20, 40)         │       200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 40)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100)            │        56,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,501 (1001.96 KB)

 Trainable params: 256,501 (1001.96 KB)

 Non-trainable params: 0 (0.00 B)

None


### Performance Metrics and Accuracy

In [54]:
y_pred = model.predict(X_test)
y_pred = np.where(y_pred > 0.5, 1, 0)

738/738 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [55]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
print(confusion_matrix(y_test, y_pred))

[[10384  1309]
 [ 1208 10707]]


In [56]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.89      0.89     11693
           1       0.89      0.90      0.89     11915

    accuracy                           0.89     23608
   macro avg       0.89      0.89      0.89     23608
weighted avg       0.89      0.89      0.89     23608



In [57]:
accuracy_score(y_test, y_pred)

0.8933835987800746